![DB Academy](./Includes/images/db-academy.png)


<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #1B5162; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lesson 06</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Working with Delta Lake</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">See what makes a Databricks table different from a plain CSV, transactional writes, full version history, time travel, and schema safety, then exercise each one against a real table.</div>
</div>

</div>

## REQUIRED: SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select SQL Warehouse Compute</strong>
  <div style="color:#333;">

This notebook requires a <strong>SQL Warehouse</strong> for execution. Before running any cells, attach this notebook to a SQL Warehouse. Select the compute tile in the top-right corner of the notebook, then choose your SQL Warehouse from the list.
  </div>
</div>

## PREREQUISITE - ENVIRONMENT SETUP

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">
    Required Setup
  </strong>
  <div style="color:#333;">
  Before running this notebook, make sure you have already executed the <strong>Required Setup</strong>, in the <strong>00 - Course Overview</strong> notebook, to properly setup your catalog, schema, and files for this notebook.
  </div>
</div>

### Setup
Run the cell below to configure your environment for this lesson. The setup script will initialize your catalog and schema so your queries run against the correct context.

In [0]:
%run ./Includes/Classroom-Setup


<!-- LEARN: What is Delta Lake -->
<!-- Template: vertical-stack + key-point callout -->

<div style="max-width: 950px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<div style="font-size: 20pt; font-weight: 700; color: #0b2026; margin-bottom: 6px;">What Makes a Databricks Table Different</div>
<div style="font-size: 14pt; color: #5A6F77; margin-bottom: 24px;">Every table you create in Databricks is a <strong>Delta Lake</strong> table by default, an open table format that turns plain Parquet files into something with database-style guarantees.</div>

<div style="display: flex; flex-direction: column; gap: 6px;">

  <!-- Top: data files -->
  <div style="background: #1C3037; color: white; border-radius: 8px 8px 4px 4px; padding: 18px 24px; text-align: center;">
    <div style="font-size: 16pt; font-weight: 700;">Data Files (Parquet)</div>
    <div style="font-size: 14pt; margin-top: 4px; opacity: 0.9;">Columnar files that hold the actual rows.</div>
  </div>

  <!-- Middle: transaction log -->
  <div style="background: #2574B5; color: white; border-radius: 4px; padding: 18px 24px; text-align: center;">
    <div style="font-size: 16pt; font-weight: 700;">Transaction Log (<code style="background:rgba(255,255,255,0.15); padding:2px 6px; border-radius:4px;">_delta_log/</code>)</div>
    <div style="font-size: 14pt; margin-top: 4px; opacity: 0.9;">A series of JSON files that record every change. This is what makes versioning and ACID guarantees possible.</div>
  </div>

  <!-- Bottom: capabilities -->
  <div style="background: #02A36F; color: white; border-radius: 4px 4px 8px 8px; padding: 18px 24px; text-align: center;">
    <div style="font-size: 16pt; font-weight: 700;">What you get on top</div>
    <div style="font-size: 14pt; margin-top: 4px; opacity: 0.9;">ACID transactions · INSERT / UPDATE / DELETE / MERGE · Time Travel · Schema Enforcement &amp; Evolution</div>
  </div>

</div>

<!-- Key point callout -->
<div style="margin-top: 20px; padding: 16px 20px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <strong>Why this matters for analysts:</strong> the same SQL you'd write against a traditional database, <code>UPDATE</code>, <code>DELETE</code>, <code>MERGE</code>, works directly on these tables. And because every change is logged, you can audit, undo, and time-travel without ever leaving SQL.
  </div>
</div>

</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

**ACID, in one minute**

Delta Lake brings the four classical database guarantees to data lake storage:

- **Atomicity**: a write either fully succeeds or fully rolls back. No half-applied changes.
- **Consistency**: data has to satisfy the table's schema and constraints, or the write fails.
- **Isolation**: concurrent writes don't see each other's intermediate state.
- **Durability**: once a transaction commits, the change is saved and survives failures.

These guarantees are what let multiple analysts and pipelines write to the same table at the same time without corrupting each other's work.

</details>

### Explore: Create a Unity Catalog managed table

Let's create a small `delta_students` table to experiment with for the rest of the lesson. The `id` column uses `GENERATED ALWAYS AS IDENTITY` so Delta auto-assigns an incrementing primary key for every row you insert.

In [0]:
CREATE OR REPLACE TABLE delta_students (
  id      BIGINT GENERATED ALWAYS AS IDENTITY,
  name    STRING,
  grade   FLOAT,
  country STRING
);

The table is empty for now, but a transaction log entry already exists for the `CREATE TABLE` operation (we'll see it shortly). Note the `OR REPLACE` clause: if you re-run the lesson, this will recreate the table cleanly so you start from a known state.

### Explore: Modify data with INSERT, UPDATE, and DELETE

UC managed tables support the standard SQL DML operations, `INSERT`, `UPDATE`, `DELETE`, and `MERGE`. Each one creates a new version of the table behind the scenes. Let's add some students.

In [0]:
INSERT INTO delta_students (name, grade, country) VALUES
  ('Lucas', 4.0, 'Italy'),
  ('Ana',   3.5, 'Germany');

SELECT * FROM delta_students ORDER BY id;

Now add three more students. These will land at the next available IDs.

In [0]:
INSERT INTO delta_students (name, grade, country) VALUES
  ('Mary',   2.0, 'Finland'),
  ('John',   3.5, 'USA'),
  ('Judith', 4.0, 'Singapore');

Now make a couple of corrections, Lucas moved, and one of Mary's grades was entered wrong. We'll then re-read the table to confirm.

In [0]:
UPDATE delta_students SET country = 'France' WHERE name = 'Lucas';
UPDATE delta_students SET grade = 2.5 WHERE id = 3;

SELECT * FROM delta_students ORDER BY id;

Lucas is now in France, and the row with `id = 3` (Mary) has `grade = 2.5`. The original values aren't lost. Delta saved them in earlier versions, which we're about to look at.


<!-- LEARN: Time Travel. DESCRIBE HISTORY and VERSION AS OF -->
<!-- Template: 2-card-colored-header-guidance -->

<div style="max-width: 950px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<div style="font-size: 20pt; font-weight: 700; color: #0b2026; margin-bottom: 6px;">Every Change is Recorded. Every Version is Queryable.</div>
<div style="font-size: 14pt; color: #5A6F77; margin-bottom: 24px;">Each <code>INSERT</code>, <code>UPDATE</code>, and <code>DELETE</code> you just ran appended a new entry to the table's transaction log. Two SQL constructs let you inspect and read those entries.</div>

<div style="display: flex; gap: 20px; justify-content: center;">

<!-- Card 1: DESCRIBE HISTORY -->
<div style="flex: 1; border: 2px solid #e0e0e0; border-radius: 12px; overflow: hidden; background: white;">
  <div style="background: #4299E0; color: white; padding: 14px 20px; text-align: center;">
    <div style="font-size: 18pt; font-weight: bold;">DESCRIBE HISTORY</div>
  </div>
  <div style="padding: 18px 20px;">
    <div style="font-size: 14pt; color: #555; line-height: 1.6; margin-bottom: 14px;">
      Shows the full change log for a table, every version number, timestamp, operation type, and who made the change.
    </div>
    <div style="background: rgba(66,153,224,0.10); border-left: 4px solid #4299E0; padding: 10px 12px; border-radius: 6px; font-size: 14pt;">
      <strong>Use when:</strong> auditing what happened to a table, debugging unexpected data, or finding a version number to restore.
    </div>
  </div>
</div>

<!-- Card 2: Time Travel -->
<div style="flex: 1; border: 2px solid #e0e0e0; border-radius: 12px; overflow: hidden; background: white;">
  <div style="background: #00A972; color: white; padding: 14px 20px; text-align: center;">
    <div style="font-size: 18pt; font-weight: bold;">VERSION AS OF / TIMESTAMP AS OF</div>
  </div>
  <div style="padding: 18px 20px;">
    <div style="font-size: 14pt; color: #555; line-height: 1.6; margin-bottom: 14px;">
      Query the table as it existed at a specific version number or point in time. The current table is unchanged. You're just reading the older snapshot.
    </div>
    <div style="background: rgba(0,169,114,0.10); border-left: 4px solid #00A972; padding: 10px 12px; border-radius: 6px; font-size: 14pt;">
      <strong>Use when:</strong> seeing what the data looked like before a change, comparing versions, or recovering accidentally modified rows.
    </div>
  </div>
</div>

</div>

</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

**How versioning works under the hood**

- Every DML or DDL statement on a UC managed table writes a new entry to its transaction log.
- Each entry is a **version**, starting at 0 (the initial `CREATE TABLE`) and incrementing by 1 per operation.
- The log records the operation type, timestamp, user, and which data files were added or removed.
- Old data files are kept around (default retention: 30 days) so older versions remain queryable. After the retention window, the `VACUUM` command can clean them up.

**Two equivalent syntaxes**

<code>SELECT * FROM delta_students VERSION AS OF 1</code> and <code>SELECT * FROM delta_students@v1</code> return the same result. Use whichever reads better in context.

</details>

### Explore: View the full change history

Let's look at the change log for `delta_students`.

In [0]:
DESCRIBE HISTORY delta_students;

You should see one row per operation, a `CREATE OR REPLACE TABLE` (version 0), two `WRITE` operations from your `INSERT` statements, and the `UPDATE` operations. Each row tells you the version number, when it happened, the operation type, and the user. Take a moment to match each row to what you ran above.

### Explore: Query an earlier version

Now let's go back in time. Read the table as it existed at version 1, right after the first `INSERT`, before the second `INSERT` and the `UPDATE`s.

In [0]:
SELECT * FROM delta_students VERSION AS OF 1 ORDER BY id;

You should see only the first two rows, Lucas in **Italy** (not France) and Ana. The current version of the table is unchanged; this is just a read of the past.

The shorthand `@v` syntax does the same thing. Use whichever you find more readable.

In [0]:
SELECT * FROM delta_students@v2 ORDER BY id;

### Explore: Compare versions side by side

A common debugging pattern is to compare two versions to see *exactly which rows changed*. The `MINUS` set operator returns rows in the first query that aren't in the second, perfect for diffing snapshots.

In [0]:
-- Rows present in version 2 but not in version 1, i.e., what the second INSERT added.
SELECT * FROM delta_students@v2
MINUS
SELECT * FROM delta_students@v1;

You should see Mary, John, and Judith, the three rows added by the second `INSERT`.

### Explore: Recover from a mistake with RESTORE

Time travel isn't just for reading the past. You can promote an old version back to the current state. Imagine a teammate accidentally drops every row.

In [0]:
DELETE FROM delta_students;
SELECT COUNT(*) AS rows_remaining FROM delta_students;

Zero rows. Bad day for our teammate, but a fixable one. Use `DESCRIBE HISTORY` to find the version number *before* the `DELETE` (it should be the second-to-last entry now), then `RESTORE` the table to it.

In [0]:
DESCRIBE HISTORY delta_students;

<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
<strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.05em;">Warning: confirm the version before you restore</strong>
<div style="color:#333;">

<code>RESTORE</code> creates a <em>new</em> version of the table whose contents match the version you point to. Pick the version <em>just before</em> the operation you want to undo. In our case, that's the version right before the <code>DELETE</code>.

</div>
</div>

In the `DESCRIBE HISTORY` output above, look for the version with operation `DELETE`. The version we want to restore to is the one immediately above it. If your DELETE is at version 5, restore version 4.

In [0]:
-- Adjust the version number below to match your history (the version just before the DELETE).
RESTORE TABLE delta_students VERSION AS OF 4;

SELECT * FROM delta_students ORDER BY id;

The table is back. All five rows return, with Lucas correctly in France and Mary's grade at 2.5, the state right before the bad `DELETE`. And the restore itself is a new entry in the history log, so the audit trail is intact.


<!-- LEARN: Schema Enforcement vs Schema Evolution -->
<!-- Template: 2-card-colored-header-guidance -->

<div style="max-width: 950px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<div style="font-size: 20pt; font-weight: 700; color: #0b2026; margin-bottom: 6px;">Schema Enforcement and Schema Evolution</div>
<div style="font-size: 14pt; color: #5A6F77; margin-bottom: 24px;">Delta Lake protects your tables from accidental schema drift, while still letting you grow them deliberately when the data changes shape.</div>

<div style="display: flex; gap: 20px; justify-content: center;">

<!-- Card 1: Enforcement -->
<div style="flex: 1; border: 2px solid #e0e0e0; border-radius: 12px; overflow: hidden; background: white;">
  <div style="background: #FF5F46; color: white; padding: 14px 20px; text-align: center;">
    <div style="font-size: 18pt; font-weight: bold;">Schema Enforcement</div>
  </div>
  <div style="padding: 18px 20px;">
    <div style="font-size: 14pt; color: #555; line-height: 1.6; margin-bottom: 14px;">
      The default behavior. Any write whose schema doesn't match the table's defined schema is <strong>rejected</strong>, wrong types, extra columns, missing required columns all fail loudly.
    </div>
    <div style="background: rgba(255,95,70,0.10); border-left: 4px solid #FF5F46; padding: 10px 12px; border-radius: 6px; font-size: 14pt;">
      <strong>Use when:</strong> you want a stable contract, downstream dashboards and reports can rely on the table's shape.
    </div>
  </div>
</div>

<!-- Card 2: Evolution -->
<div style="flex: 1; border: 2px solid #e0e0e0; border-radius: 12px; overflow: hidden; background: white;">
  <div style="background: #00A972; color: white; padding: 14px 20px; text-align: center;">
    <div style="font-size: 18pt; font-weight: bold;">Schema Evolution</div>
  </div>
  <div style="padding: 18px 20px;">
    <div style="font-size: 14pt; color: #555; line-height: 1.6; margin-bottom: 14px;">
      An opt-in mode that lets a write <strong>add new columns</strong> to the table automatically. The existing schema still enforces; the new shape just expands.
    </div>
    <div style="background: rgba(0,169,114,0.10); border-left: 4px solid #00A972; padding: 10px 12px; border-radius: 6px; font-size: 14pt;">
      <strong>Use when:</strong> upstream data adds new fields and you want them captured without manually altering the table first.
    </div>
  </div>
</div>

</div>

</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

Schema Evolution is opt-in per write. You turn it on with `mergeSchema = true` (DataFrame writes) or `WITH SCHEMA EVOLUTION` (SQL `MERGE` statements). Outside of that explicit opt-in, enforcement always wins. This is what protects production tables from a bad CSV silently adding a `customerID2` column nobody asked for.

Docs:
<a href="https://docs.databricks.com/aws/en/delta/update-schema" target="_blank">AWS</a> |
<a href="https://learn.microsoft.com/azure/databricks/delta/update-schema" target="_blank">Azure</a> |
<a href="https://docs.databricks.com/gcp/en/delta/update-schema" target="_blank">GCP</a>

</details>


<!-- Micro-win summary -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="margin-top: 10px; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">What you just did:</div>
    <ul style="padding-left: 20px; margin: 0;">
      <li>Created a UC managed table with an auto-incrementing identity column.</li>
      <li>Modified rows with <code>INSERT</code> and <code>UPDATE</code>, each operation captured atomically by the transaction log.</li>
      <li>Inspected the change log with <code>DESCRIBE HISTORY</code>.</li>
      <li>Read previous snapshots with <code>VERSION AS OF</code> and the <code>@v</code> shorthand, and diffed two versions with <code>MINUS</code>.</li>
      <li>Recovered from an accidental <code>DELETE</code> by promoting an earlier version back to current with <code>RESTORE TABLE</code>.</li>
      <li>Saw how Schema Enforcement and Schema Evolution work together to keep tables both safe and adaptable.</li>
    </ul>
    
  </div>
</div>
</div>


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>